**calc95pct.ipynb**
- Calculates mean daily temperature (tas) climatology for 1979-2000 using AUS-11 (BARRA-R2).

**Reads:** "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/tas/latest/" \
**Writes:** "ID_HW_BARRA/data/preprocess/t95_baseline.nc" \
**Compute:** xxlarge (28CPU, 126GB) \
**Environment:** analysis3

In [1]:
import xarray as xr, netCDF4 as nc, numpy as np, pandas as pd, os
from pathlib import Path

import dask
import dask.array as da
from dask.distributed import LocalCluster, Client, wait
from datetime import datetime

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml


In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
workingDir = Path().absolute()
print(f"{workingDir}")

/g/data/ng72/ms5578/ID_HW_BARRA


In [3]:
client = Client()
client

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that i

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 8
Total threads: 48,Total memory: 250.26 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39209,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:36155,Total threads: 6
Dashboard: http://127.0.0.1:42965/status,Memory: 31.28 GiB
Nanny: tcp://127.0.0.1:32859,


In [4]:
tas_path = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/tas/latest/"
write_path = f'{workingDir}/data/preprocess/'

In [5]:
sdate, edate ='19790101', '20001231'

In [6]:
fdates = [m.strftime('%Y%m') for m in pd.date_range(sdate, edate, freq='ME')]
fnames = [s for s in os.listdir(tas_path) if any(f in s for f in fdates)]
fpaths = sorted([tas_path + f for f in fnames])

# Open hourly tas
tas_ds = xr.open_mfdataset(
    fpaths,
    concat_dim='time',
    combine='nested',
    parallel=True,
    data_vars='minimal',
    coords='minimal',
    drop_variables='time_bnds',
    chunks='auto',
)

# Derive daily Tmax/Tmin from hourly tas, then daily mean (tmax+tmin)/2
tas = tas_ds.tas
tas_daily_max = tas.resample(time='1D').max()
tas_daily_min = tas.resample(time='1D').min()
tas_daily_mean = (tas_daily_max + tas_daily_min) / 2.0
tas_daily = tas_daily_mean.to_dataset(name='tas')

datestr = f"s{sdate}_e{edate}"

# 95th percentile of daily mean temperature over time
t95 = tas_daily.reduce(np.nanpercentile, q=95, dim='time')
t95 = t95.rename(name_dict={'tas': 'PRCTILE95'})

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/pyproj/__init__.py:91: UserWarning: Valid PROJ data directory not found. Either set the path using the environmental variable PROJ_DATA (PROJ 9.1+) | PROJ_LIB (PROJ<9.1) or with `pyproj.datadir.set_data_dir`.
  warnings.warn(str(err))
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/pyproj/__init__.py:91: UserWarning: Valid PROJ data directory not found. Either set the path using the environmental variable PROJ_DATA (PROJ 9.1+) | PROJ_LIB (PROJ<9.1) or with `pyproj.datadir.set_data_dir`.
  warnings.warn(str(err))
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/pyproj/__init__.py:91: UserWarning: Valid PROJ data directory not found. Either set the path using the environmental variable PROJ_DATA (PROJ 9.1+) | PROJ_LIB (PROJ<9.1) or with `pyproj.datadir.set_data_dir`.
  warnings.warn(str(err))
/g/data/xp65/public/apps/med_conda/envs/

In [7]:
# Minimal fix 1: clear stale encodings
try:
    t95.encoding.clear()
    for v in t95.data_vars:
        t95[v].encoding.clear()
except Exception:
    pass

encoding = {"PRCTILE95":{"zlib": True, "complevel": 4, "shuffle": True}}

In [ ]:
t95 = t95.persist()
wait(t95)           # ensure reduction finished on workers
t95 = t95.compute() # materialize to memory to avoid 'tas' backend refs
tas_ds.close()

t95.to_netcdf(f'{write_path}t95_baseline.nc',
              encoding=encoding,
              engine='netcdf4')